# HRF-POL-B41L-01 — polarização sobre B41L congelada (v0.1.1)

[M] Campo real de arestas, operadores Whitney e kernel de Verlet originais. V1 principal, V0 pareado; V1E não calibra. Investiga W, não valida G/D, fóton individual ou PHOTON-CLOSURE-F.

[D] **A rodada científica completa NÃO foi executada na distribuição.** Os exemplos incluídos são fumaça. G1/P01–P05 originais permanecem PENDING porque POL-01 não foi fornecido. O controle Jones novo não fecha o HRF. A célula 11 exige `CONFIRMO_COMPLETA=True` e só deve ser acionada após revisão do pré-registro e dos gates de fumaça. Critérios não podem ser ajustados após observar resultados.


## 2. Montar Drive (opcional; upload manual continua disponível)

In [ ]:
from pathlib import Path
import os,sys,json,subprocess,shutil,zipfile,hashlib,tempfile,datetime
from google.colab import drive,files
USAR_DRIVE=True
if USAR_DRIVE:
    drive.mount('/content/drive')
DRIVE_ROOT=Path('/content/drive/MyDrive')


## 3. Localizar ou enviar a distribuição e a dependência B41L

In [ ]:
PROJECT=Path('/content/HRF_POL_B41L_01')
if not (PROJECT/'scripts/pol.py').is_file():
    locations=[Path('/content'),DRIVE_ROOT/'HRF_POL_B41L_01',DRIVE_ROOT/'PHYSICS/00-Inbox']
    candidates=[p for folder in locations if folder.exists() for p in folder.glob('HRF_POL_B41L_01_DISTRIBUICAO*.zip')]
    if not candidates:
        print('Selecione o ZIP de distribuição HRF_POL_B41L_01_DISTRIBUICAO, que inclui o notebook e os fontes.')
        uploaded=files.upload()
        candidates=[Path('/content')/name for name in uploaded if name.endswith('.zip')]
    if not candidates:
        raise FileNotFoundError('Distribuição não fornecida; o ZIP de resultados B41L isolado não contém o programa novo.')
    valid=[]
    for candidate in candidates:
        with zipfile.ZipFile(candidate) as testzip:
            if 'HRF_POL_B41L_01/scripts/pol.py' in testzip.namelist():valid.append(candidate)
    if not valid:raise FileNotFoundError('Selecione o ZIP da distribuição HRF-POL-B41L-01, não só o ZIP B41L.')
    bundle=max(valid,key=lambda p:p.stat().st_mtime)
    with zipfile.ZipFile(bundle) as z:
        if z.testzip() is not None:raise ValueError('ZIP corrompido')
        for item in z.infolist():
            destination=(Path('/content')/item.filename).resolve()
            if not destination.is_relative_to(Path('/content')) or ((item.external_attr>>16)&0o170000)==0o120000:
                raise ValueError('Caminho inseguro no ZIP')
        z.extractall('/content')
if not (PROJECT/'scripts/pol.py').is_file():raise FileNotFoundError('Raiz esperada /content/HRF_POL_B41L_01 não encontrada')
B41L=PROJECT/'dependencias/HRF_B41L_B1_Lancamento_Limpo'
if not B41L.exists():
    print('Selecione o ZIP ORIGINAL de fontes/pré-registro B41L (não o ZIP apenas de resultados).')
    uploaded=files.upload()
    for name in uploaded:
        if not name.endswith('.zip'):continue
        shutil.copy2(name,PROJECT/'dependencias/B41L_fonte_original.zip')
        with zipfile.ZipFile(name) as z:
            for item in z.infolist():
                destination=(PROJECT/'dependencias'/item.filename).resolve()
                if not destination.is_relative_to((PROJECT/'dependencias').resolve()):raise ValueError('ZIP inseguro')
            z.extractall(PROJECT/'dependencias')
print('Projeto:',PROJECT)


## 4. Verificação de integridade antes de importar o solver

In [ ]:
def verificar_manifesto(path,root):
    count=0
    for line in path.read_text(encoding='utf-8').splitlines():
        if not line or line.startswith('#'):continue
        expected,name=line.split('  ',1);p=(root/name).resolve()
        if not p.is_relative_to(root.resolve()) or not p.is_file():raise ValueError('Arquivo ausente/inseguro: '+name)
        h=hashlib.sha256()
        with p.open('rb') as f:
            for block in iter(lambda:f.read(1048576),b''):h.update(block)
        if h.hexdigest()!=expected:raise ValueError('SHA-256 difere: '+name)
        count+=1
    return count
print('Entrega:',verificar_manifesto(PROJECT/'MANIFESTO_SHA256.txt',PROJECT),'arquivos íntegros')
print('Pré-registro:',verificar_manifesto(PROJECT/'HASHES_PREREGISTRO.txt',PROJECT),'arquivos íntegros')
print('B41L:',verificar_manifesto(B41L/'HRF_B41L_hashes_antes_da_execucao_2026-10-29_2000-clau.txt',B41L),'arquivos originais íntegros')


## 5. Dependências e registro do ambiente real

In [ ]:
USAR_VERSOES_AUDITADAS_B41L=True
wanted=['numpy==2.5.3','scipy==1.18.1','matplotlib>=3.7']
probe=subprocess.run([sys.executable,'-m','pip','install','--dry-run',*wanted],capture_output=True,text=True) if USAR_VERSOES_AUDITADAS_B41L else None
if probe is not None and probe.returncode==0:
    subprocess.run([sys.executable,'-m','pip','install',*wanted],check=True)
else:
    print('Versões históricas indisponíveis/incompatíveis neste runtime; usando requisitos compatíveis. Não se afirma equivalência binária.')
    if probe:print(probe.stderr[-2500:])
    subprocess.run([sys.executable,'-m','pip','install','-r',str(PROJECT/'requirements.txt')],check=True)
# Os comandos científicos rodam em processos novos, para não misturar NumPy já importado no kernel.
for key in ('OPENBLAS_NUM_THREADS','OMP_NUM_THREADS','MKL_NUM_THREADS'):os.environ[key]='1'
os.environ['PYTHONDONTWRITEBYTECODE']='1'
OUTPUT_ROOT=(DRIVE_ROOT/'HRF_POL_B41L_01/execucao_v0_1_1') if USAR_DRIVE else Path('/content/HRF_POL_B41L_01_SAIDAS')
OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
def executar(*args):
    command=[sys.executable,'-I',str(PROJECT/'scripts/pol.py'),*map(str,args)]
    with (OUTPUT_ROOT/'execucao_notebook.log').open('a',encoding='utf-8') as log:
        log.write(json.dumps(dict(UTC=datetime.datetime.now(datetime.timezone.utc).isoformat(),command=command))+'\n')
        process=subprocess.Popen(command,cwd=PROJECT,env=os.environ.copy(),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in process.stdout:
            print(line,end='');log.write(line);log.flush()
        if process.wait()!=0:raise RuntimeError('Comando falhou; confira o log preservado.')
print('Python efetivo:',sys.version)
print('Python3.13.16/NumPy2.5.3/SciPy1.18.1 são referências históricas, não versões presumidas.')


## 6. Testes de infraestrutura e álgebra (incluem referência pequena de fidelidade)

In [ ]:
stamp=datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
executar('verify','--output',OUTPUT_ROOT/'integridade.json')
executar('test','--output',OUTPUT_ROOT/('testes_'+stamp))


## 7. Controle Maxwell/Jones novo, independente; G1 original permanece PENDING

In [ ]:
executar('controls','--output',OUTPUT_ROOT/'controle_Jones_novo.json')


## 8. Contrato dos observáveis físicos

In [ ]:
print((PROJECT/'RELATORIO_MODELO.md').read_text(encoding='utf-8'))
print('A reconstrução usa E=-Wv e B=curlWq reais, gauge temporal residual e fasores de séries físicas.')
print('As rotinas congeladas são importadas; o kernel AST extraído é verificado sem editar B41L.')


## 9. Fumaça pequena — seis casos, não certifica a física

In [ ]:
SMOKE=OUTPUT_ROOT/'fumaca_v0_1_1'
executar('plan','--perfil','fumaca','--output',SMOKE)
executar('run','--perfil','fumaca','--output',SMOKE,'--executar')


## 10. Avaliar fumaça e revisar gates

In [ ]:
executar('evaluate','--perfil','fumaca','--output',SMOKE)
smoke_status=json.loads((SMOKE/'summary.json').read_text())
print('Não alterar limiares para transformar PENDING/FAIL em PASS. A duração de fumaça pode não resolver a janela4ciclos.')
print(json.dumps(smoke_status,indent=2,ensure_ascii=False))


## 11. Rodada completa — desligada por padrão; exige decisão explícita

In [ ]:
CONFIRMO_COMPLETA=False
COMPLETE=OUTPUT_ROOT/'completo_v0_1_1'
executar('plan','--perfil','completo','--output',COMPLETE)
if CONFIRMO_COMPLETA:
    print('Execução explícita do pré-registro; G1 original continuará PENDING se POL-01 não tiver sido incorporado em nova versão.')
    executar('run','--perfil','completo','--output',COMPLETE,'--executar','--confirmo-completa')
else:
    print('NOT_EXECUTED — nenhuma rodada completa iniciada.')


## 12. Avaliação científica — só dos casos realmente concluídos

In [ ]:
if CONFIRMO_COMPLETA:
    executar('evaluate','--perfil','completo','--output',COMPLETE)
    print((COMPLETE/'summary.json').read_text())
else:
    print('Avaliação completa NOT_EXECUTED. Fumaça e Jones não autorizam PASS relacional.')


## 13. Gráficos, tabela e relatório dos dados existentes

In [ ]:
DATA=COMPLETE if CONFIRMO_COMPLETA else SMOKE
PLOTS=OUTPUT_ROOT/('figuras_completa' if CONFIRMO_COMPLETA else 'figuras_fumaca')
executar('figures','--input',DATA,'--output',PLOTS)
print((PLOTS/'RELATORIO_DADOS.md').read_text())


## 14. Compactação da entrega, sem apagar dados e com teste de integridade

In [ ]:
stamp=datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
ARCHIVE=OUTPUT_ROOT.parent/('HRF_POL_B41L_01_RESULTADOS_'+stamp+'.zip')
with zipfile.ZipFile(ARCHIVE,'x',zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for p in OUTPUT_ROOT.rglob('*'):
        if p.is_file():z.write(p,'execucao/'+p.relative_to(OUTPUT_ROOT).as_posix())
    for folder in ('src','scripts','configs','documentos'):
        for p in (PROJECT/folder).rglob('*'):
            if p.is_file() and '__pycache__' not in p.parts:z.write(p,'projeto/'+p.relative_to(PROJECT).as_posix())
    for name in ('HASHES_PREREGISTRO.txt','PREREGISTRO_HRF_POL_B41L_01.md','EMENDA_PREEXECUCAO_01.md','RELATORIO_MODELO.md'):
        z.write(PROJECT/name,'projeto/'+name)
    z.write(PROJECT/'dependencias/B41L_fonte_original.zip','dependencias/B41L_fonte_original.zip')
    z.write(PROJECT/'dependencias/B41L_resultados_originais.zip','dependencias/B41L_resultados_originais.zip')
with zipfile.ZipFile(ARCHIVE) as z:
    if z.testzip() is not None:raise ValueError('Falha CRC no ZIP')
h=hashlib.sha256()
with ARCHIVE.open('rb') as f:
    for block in iter(lambda:f.read(1048576),b''):h.update(block)
print('ZIP:',ARCHIVE,'bytes:',ARCHIVE.stat().st_size,'SHA256:',h.hexdigest())


## 15. Salvar no Drive e oferecer download

In [ ]:
if USAR_DRIVE:
    destination=DRIVE_ROOT/'HRF_POL_B41L_01/entregas'/ARCHIVE.name
    destination.parent.mkdir(parents=True,exist_ok=True)
    if destination.exists():raise FileExistsError('Destino já existe; preserve-o e escolha outro nome.')
    shutil.copy2(ARCHIVE,destination)
    print('Cópia salva:',destination)
    print('Drive: https://drive.google.com/drive/u/0/my-drive — abra HRF_POL_B41L_01/entregas')
from IPython.display import display,FileLink
display(FileLink(str(ARCHIVE)))
BAIXAR_PELO_NAVEGADOR=False  # ZIPs grandes podem travar o navegador; prefira a cópia no Drive.
if BAIXAR_PELO_NAVEGADOR:files.download(str(ARCHIVE))
